# Step 5: Fine-Tuning EoMT with LoRA — Three Experiments

All three experiments use `MaskClassificationLoRA` and the same LoRA config.
The only difference between them is which parameters have `requires_grad=True`.
`configure_optimizers` adapts automatically: it filters by `requires_grad` and
routes whatever is trainable into the right param groups for
`TwoStageWarmupPolySchedule` + LLRD.

| Experiment | Trainable |
|---|---|
| 1 — head only | prediction head; all LoRA adapters frozen |
| 2 — decoder LoRA | prediction head + LoRA adapters in blocks 9–11 |
| 3 — full LoRA | prediction head + LoRA adapters in all 12 blocks (LLRD) |

In [ ]:
!pip install lightning gitignore_parser peft > /dev/null
!pip install -U "torchao>=0.16.0" > /dev/null
!pip install -U 'jsonargparse[signatures]>=4.27.7' > /dev/null
!pip install wandb > /dev/null

In [ ]:
from google.colab import drive, userdata
import os, sys, json, yaml, glob, shutil
import torch
import torch.nn.functional as F
import wandb
from tqdm import tqdm
from lightning import seed_everything
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor
from lightning.pytorch.loggers import WandbLogger

In [ ]:
import torch.serialization
_orig_load = torch.serialization.load
def _patched_load(*args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return _orig_load(*args, **kwargs)
torch.load = _patched_load

In [ ]:
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

project_root = '/content/drive/MyDrive/FundGitHubProject'
eomt_folder  = project_root + '/eomt'

if not os.path.exists('/content/ProjectFolder'):
    os.symlink(project_root, '/content/ProjectFolder')

os.chdir(project_root)
for p in [project_root, eomt_folder]:
    if p not in sys.path:
        sys.path.insert(0, p)

Mounted at /content/drive


In [ ]:
from eval.iouEval import iouEval
from training.mask_classification_panoptic import MaskClassificationPanoptic
from training.mask_classification_lora import MaskClassificationLoRA
from models.eomt import EoMT
from models.vit import ViT
from datasets.cityscapes_semantic import CityscapesSemantic
import torch

torch.serialization.add_safe_globals([EoMT])

seed_everything(0, verbose=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [ ]:
drive_data = os.path.join(eomt_folder, 'data')
local_data = '/content/cityscapes_data'
if not os.path.exists(local_data):
    print("Copying dataset to local storage...")
    shutil.copytree(drive_data, local_data)
    print("Done.")
data_path = local_data

Copying dataset to local storage...
Done.


## 2. Zero-Shot Baseline

In [ ]:
coco_cfg_path = os.path.join(eomt_folder, 'configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml')
with open(coco_cfg_path) as f:
    coco_config = yaml.safe_load(f)

bin_path = os.path.join(eomt_folder, 'eomt_weights/eomt_coco.bin')

encoder = ViT(img_size=640, backbone_name="vit_base_patch14_reg4_dinov2")
network = EoMT(num_classes=133, encoder=encoder, num_q=200,
               num_blocks=3, masked_attn_enabled=False)
model_coco = MaskClassificationPanoptic(
    network=network,
    img_size=(640, 640),
    num_classes=133,
    stuff_classes=coco_config["data"].get("init_args", {}).get("stuff_classes", []),
    attn_mask_annealing_enabled=False,
)
ckpt = torch.load(bin_path, map_location="cpu")
model_coco.load_state_dict(ckpt.get("state_dict", ckpt), strict=False)
model_coco.to(device).eval()
print("COCO model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.


COCO model loaded.


In [ ]:
map_file = os.path.join(project_root, 'coco-classes-mapping-master/coco_mapping_80to91.json')
with open(map_file) as f:
    coco_idx_map = {int(k)-1: int(v) for k, v in json.load(f).items()}

things_map = {1:11, 2:18, 3:13, 4:17, 6:15, 7:16, 8:14, 10:6, 13:7}
stuff_map  = {100:0, 123:1, 91:2, 129:2, 109:3, 110:3, 111:3,
              112:3, 131:3, 117:4, 116:8, 125:8, 126:9, 119:10}

def bridge_to_cs(pred_tensor):
    res = torch.full_like(pred_tensor, 19)
    for idx, coco_id in coco_idx_map.items():
        if coco_id in things_map:
            res[pred_tensor == idx] = things_map[coco_id]
    for stuff_id, cs_id in stuff_map.items():
        res[pred_tensor == stuff_id] = cs_id
    return res

In [ ]:
dm_zs = CityscapesSemantic(path=data_path, batch_size=1, num_workers=2)
dm_zs.setup()

ev = iouEval(20)
for batch in tqdm(dm_zs.val_dataloader(), desc="zero-shot"):
    imgs, targets = batch
    gt = model_coco.to_per_pixel_targets_semantic(targets, 19)[0].to(device)
    with torch.no_grad():
        tx  = model_coco.resize_and_pad_imgs_instance_panoptic([imgs[0].to(device)])
        mp, cp = model_coco(tx)
        mp  = model_coco.revert_resize_and_pad_logits_instance_panoptic(
                  F.interpolate(mp[-1], model_coco.img_size, mode="bilinear"),
                  [imgs[0].shape[-2:]])
        pred = model_coco.to_per_pixel_preds_panoptic(
                   mp, cp[-1], model_coco.stuff_classes, 0.8, 0.8)[0][..., 0]
    ev.addBatch(bridge_to_cs(pred).unsqueeze(0).unsqueeze(0),
                gt.unsqueeze(0).unsqueeze(0))

_, ious = ev.getIoU()
zeroshot_miou = ious[:19].mean().item() * 100
print(f"Zero-shot mIoU: {zeroshot_miou:.2f}%")

zero-shot: 100%|██████████| 500/500 [01:54<00:00,  4.37it/s]

Zero-shot mIoU: 45.91%


## 3a. Architecture and Freezing Strategy

### How EoMT works

The ViT encoder processes image patches through 12 transformer blocks.
The key insight of EoMT is that the **last 3 blocks serve as the mask decoder**:
learnable query tokens are prepended to the patch token sequence at block 9,
and those final 3 blocks process image tokens and query tokens together via
self-attention.  The output query tokens are then fed into the prediction head.

```
 IMAGE (640×640)
      │
 patch_embed + pos_embed
      │
 ┌────┴──────────────────────────────────────────────────────────────┐
 │  block  0   [ qkv · fc1 · fc2 ]   ← LoRA targets                 │
 │  block  1   [ qkv · fc1 · fc2 ]                                   │
 │  block  2   [ qkv · fc1 · fc2 ]                                   │
 │  block  3   [ qkv · fc1 · fc2 ]        ENCODER                   │
 │  block  4   [ qkv · fc1 · fc2 ]                                   │
 │  block  5   [ qkv · fc1 · fc2 ]                                   │
 │  block  6   [ qkv · fc1 · fc2 ]                                   │
 │  block  7   [ qkv · fc1 · fc2 ]                                   │
 │  block  8   [ qkv · fc1 · fc2 ]                                   │
 │ ╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌ │
 │   ← query tokens (q) prepended here                               │
 │                                                                    │
 │  block  9   [ qkv · fc1 · fc2 ]   ← image + query tokens          │
 │  block 10   [ qkv · fc1 · fc2 ]        DECODER                   │
 │  block 11   [ qkv · fc1 · fc2 ]                                   │
 └────────────────────────────┬──────────────────────────────────────┘
            image tokens      │      query tokens (200 × 768)
                 │            │            │
            upscale     class_head    mask_head
           (CNN ↑)     Linear(768,20)  MLP(768,32)
                 │            │            │
                 └────────────┴────────────┘
                          einsum dot-product
                               │
                    per-pixel class logits (B, 19, H, W)
```

---

### What is frozen and what trains in each experiment

LoRA adds two small matrices **A** and **B** alongside each target weight **W**.
The effective weight becomes **W + B·A** (with B initialised to 0, so it starts
as a no-op).  Original weights **W** are always frozen; only **A** and **B** update.

```
 COMPONENT            │ Exp 1         │ Exp 2          │ Exp 3
                       │ head only     │ decoder LoRA   │ full LoRA
 ──────────────────────┼───────────────┼────────────────┼──────────────────
 patch_embed / pos_emb │ ❄  frozen     │ ❄  frozen      │ ❄  frozen
 ──────────────────────┼───────────────┼────────────────┼──────────────────
 block  0  (W)         │ ❄  frozen     │ ❄  frozen      │ ❄  frozen
           (LoRA A·B)  │ ❄  frozen     │ ❄  frozen      │ ✦  lr=3.1e-5
 block  1  (W)         │ ❄  frozen     │ ❄  frozen      │ ❄  frozen
           (LoRA A·B)  │ ❄  frozen     │ ❄  frozen      │ ✦  lr=3.5e-5
 block  2  (W)         │ ❄  frozen     │ ❄  frozen      │ ❄  frozen
           (LoRA A·B)  │ ❄  frozen     │ ❄  frozen      │ ✦  lr=3.9e-5
 block  3  (W)         │ ❄  frozen     │ ❄  frozen      │ ❄  frozen
           (LoRA A·B)  │ ❄  frozen     │ ❄  frozen      │ ✦  lr=4.3e-5
 block  4  (W)         │ ❄  frozen     │ ❄  frozen      │ ❄  frozen
           (LoRA A·B)  │ ❄  frozen     │ ❄  frozen      │ ✦  lr=4.8e-5
 block  5  (W)         │ ❄  frozen     │ ❄  frozen      │ ❄  frozen
           (LoRA A·B)  │ ❄  frozen     │ ❄  frozen      │ ✦  lr=5.3e-5
 block  6  (W)         │ ❄  frozen     │ ❄  frozen      │ ❄  frozen
           (LoRA A·B)  │ ❄  frozen     │ ❄  frozen      │ ✦  lr=5.9e-5
 block  7  (W)         │ ❄  frozen     │ ❄  frozen      │ ❄  frozen
           (LoRA A·B)  │ ❄  frozen     │ ❄  frozen      │ ✦  lr=6.6e-5
 block  8  (W)         │ ❄  frozen     │ ❄  frozen      │ ❄  frozen
           (LoRA A·B)  │ ❄  frozen     │ ❄  frozen      │ ✦  lr=7.3e-5
 ╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌
 block  9  (W)         │ ❄  frozen     │ ❄  frozen      │ ❄  frozen
           (LoRA A·B)  │ ❄  frozen     │ ✦  lr=8.1e-5   │ ✦  lr=8.1e-5
 block 10  (W)         │ ❄  frozen     │ ❄  frozen      │ ❄  frozen
           (LoRA A·B)  │ ❄  frozen     │ ✦  lr=9.0e-5   │ ✦  lr=9.0e-5
 block 11  (W)         │ ❄  frozen     │ ❄  frozen      │ ❄  frozen
           (LoRA A·B)  │ ❄  frozen     │ ✦  lr=1.0e-4   │ ✦  lr=1.0e-4
 ──────────────────────┼───────────────┼────────────────┼──────────────────
 q  (query embeddings) │ ✦  lr=1.0e-4  │ ✦  lr=1.0e-4   │ ✦  lr=1.0e-4
 class_head            │ ✦  lr=1.0e-4  │ ✦  lr=1.0e-4   │ ✦  lr=1.0e-4
 mask_head             │ ✦  lr=1.0e-4  │ ✦  lr=1.0e-4   │ ✦  lr=1.0e-4
 upscale               │ ✦  lr=1.0e-4  │ ✦  lr=1.0e-4   │ ✦  lr=1.0e-4
 ──────────────────────┼───────────────┼────────────────┼──────────────────
 Trainable params      │ ~7 M          │ ~7.15 M        │ ~7.59 M
```

`❄ frozen` = `requires_grad=False`, AdamW allocates no state, no update ever.  
`✦ trainable` = `requires_grad=True`, enters the optimizer with the LR shown.  
LR values shown are the **base LR** before the poly schedule scales them down.
The LoRA original weights **W** are always frozen — only the adapters **A** and **B** train.

## 3b. Parameter Choices and Motivation

All parameters live in the YAML files under `configs/experiments/`.
Edit those files to change a value — nothing else needs to touch the notebook.

---

### Learning rate — `lr: 1.0e-4`

`lr` is the **base LR** for the prediction head and for block 11 LoRA adapters
(the last block gets LLRD exponent 0, so no decay).

`1e-4` is the standard starting point for AdamW fine-tuning of ViT-based models
and is consistent with the EoMT authors' own Cityscapes training config.
Higher values (e.g. `3e-4`) tend to overshoot on a small dataset like Cityscapes;
lower values (e.g. `3e-5`) converge too slowly for a 10-epoch run.

---

### Layer-wise Learning Rate Decay — `llrd: 0.9`

```
block  0  LR = 1e-4 × 0.9^11 ≈ 3.1e-5   spread × 3.2 (current)
block  0  LR = 1e-4 × 0.8^11 ≈ 0.9e-5   spread × 11.6  ← too aggressive
block  0  LR = 1e-4 × 0.95^11 ≈ 5.7e-5  spread × 1.8   ← too uniform
```

`0.9` gives a **3× spread** between the lowest and highest LoRA LR.
This is enough to protect early-block features without making them learn so
slowly that 10 epochs are insufficient.  `0.8` is suitable for full fine-tuning
where the entire weight matrix is updated; for LoRA the adapters are already
a small perturbation, so a gentler decay is appropriate.

---

### LoRA rank — `lora_r: 8`

Rank controls the expressiveness of each adapter (`W_update = B·A`, where
`B ∈ R^{d×r}` and `A ∈ R^{r×d}`).

| r | adapter params per layer | notes |
|---|---|---|
| 4 | 2 × 768 × 4 = 6 144 | too small for a domain shift |
| **8** | **2 × 768 × 8 = 12 288** | **good balance — use this** |
| 16 | 2 × 768 × 16 = 24 576 | doubles params, marginal gain for 10 epochs |

`r=8` follows the LoRA paper's recommendation for medium-scale adaptation tasks.
The domain shift from COCO to Cityscapes road scenes is real but not extreme
(both contain street objects), so a higher rank is unlikely to be necessary.

---

### LoRA scaling — `lora_alpha: 16`

The adapter output is multiplied by `alpha / r` before being added to the frozen
weight.  With `alpha=16, r=8` the scaling factor is **2**.

This doubles the effective learning signal from the adapters, compensating for
the reduced capacity from using rank 8 instead of the full matrix.  The
convention `alpha = 2*r` is a common default recommended in the LoRA literature.

---

### LoRA dropout — `lora_dropout: 0.05`

Cityscapes training has ~2975 images — small enough that light regularisation
helps.  `0.05` applies a 5% dropout inside each adapter, providing a mild
regularisation effect without interfering with convergence.  Setting it to `0`
is fine if you observe underfitting; setting it above `0.1` risks slowing down
convergence on a small dataset.

---

### Warmup steps — `warmup_steps: [125, 125]`

```
raw batches / epoch     = ceil(2975 / 4)     = 744
optimizer steps / epoch = ceil(744 / 4)      = 186   ← with accumulate_grad_batches=4
total optimizer steps   = 186 × 10 epochs    = 1860

[ 93, 186]  →  head 0.50 ep,  backbone 1.00 ep,  total 15.0%
[125, 125]  →  head 0.67 ep,  backbone 0.67 ep,  total 13.4%  ← current
[186, 186]  →  head 1.00 ep,  backbone 1.00 ep,  total 20.0%
```

The **non_vit_warmup** (first number) gives the randomly-initialised head time
to produce reasonable gradients before the backbone sees any update.
The **vit_warmup** (second number) ramps the backbone LR from 0 to its LLRD-scaled
target smoothly.

`[125, 125]` keeps total warmup at 13.4% of 1860 optimizer steps, within the
recommended 10–20% window.  If you change `max_epochs`, `batch_size`, or
`accumulate_grad_batches`, recompute using the formula above.

---

### Gradient accumulation — `accumulate_grad_batches: 4`

The original EoMT Cityscapes training uses **effective batch size 16**.
A T4 GPU on Colab cannot fit 16 images at 640×640 in a single forward pass,
so we split the batch:

```
batch_size              = 4   (actual GPU memory per step)
accumulate_grad_batches = 4   (steps before optimizer.step())
──────────────────────────────
effective batch size    = 16  (matches the original training)
```

Lightning accumulates gradients over 4 steps and calls `optimizer.step()` only
on the fourth step.  From the optimizer's perspective, training looks identical
to using batch size 16 directly.

**Critical side effect on warmup_steps.**
Accumulation reduces the number of optimizer steps per epoch:

```
raw batches / epoch     = ceil(2975 / 4)      = 744
optimizer steps / epoch = ceil(744 / 4)        = 186   ← not 744
total optimizer steps   = 186 × 10 epochs      = 1860  ← not 7440
```

`TwoStageWarmupPolySchedule` receives `total_steps = estimated_stepping_batches`,
which Lightning sets to the **optimizer step count** (already divided by
`accumulate_grad_batches`).  So `warmup_steps` must also be expressed in
optimizer steps:

```
[500, 500] without accumulation → 13.4% of 7440 steps  ✓ correct
[500, 500] with  accumulation   → 53.8% of 1860 steps  ✗ nearly all training is warmup!

[125, 125] with  accumulation   → 13.4% of 1860 steps  ✓ same proportion
```

**The rule:** `warmup_steps = round(target_epoch_fraction × optimizer_steps_per_epoch)`.
Always recompute when you change `batch_size`, `accumulate_grad_batches`, or `max_epochs`.
---

### Gradient clipping — `gradient_clip_val: 0.01/0.05`

The global gradient L2 norm is clipped to `0.01/0.005` after every backward pass.
This value is intentionally tight: at step 0 the class head is random and can
produce gradients 10–100× larger than the converged backbone gradients.
Without clipping, those spikes would bypass the warmup schedule and corrupt
the pre-trained backbone weights in the very first steps.

If you notice the loss not decreasing in the first epoch, try loosening to
`0.05`; if you see NaN loss at step 0, tighten to `0.005`.

## 3. Setup

### Gradient clipping — where it lives and how to change it

Lightning applies `torch.nn.utils.clip_grad_norm_` after every backward pass
and before the optimizer step.  It computes the global L2 norm of all trainable
parameter gradients and scales every gradient down proportionally if that norm
exceeds the threshold.

**Where it is set:** `build_trainer` reads `gradient_clip_val` from the YAML:

```yaml
gradient_clip_val: 0.5   # in each experiment YAML
```

To change it, edit that line in the YAML.  `0.5` (the same default as
`main.py`) is deliberately tight — the randomly-initialised class head
produces large gradients in early steps and without clipping those can
destabilise the pre-trained backbone before the warmup schedule has had a
chance to ramp up the backbone LR gradually.

---

### Warmup steps — the arithmetic

`warmup_steps: [non_vit_warmup, vit_warmup]` are step counts, not epochs.
The formula to convert from epochs:

```
steps_per_epoch = ceil(dataset_size / batch_size)
non_vit_warmup  = round(head_warmup_epochs  * steps_per_epoch)
vit_warmup      = round(backbone_ramp_epochs * steps_per_epoch)
```

For this setup (Cityscapes ≈ 2975 training images, `batch_size=4`):

```
raw batches / epoch     = ceil(2975 / 4)  = 744
optimizer steps / epoch = ceil(744 / 4)   = 186   (accumulate_grad_batches=4)
total optimizer steps   = 186 × 10        = 1860
```

| head warmup | backbone ramp | warmup_steps | % of total |
|---|---|---|---|
| 0.5 ep | 0.5 ep | [93, 93] | 10% |
| 0.67 ep | 0.67 ep | [125, 125] | 13% ← current |
| 1.0 ep | 1.0 ep | [186, 186] | 20% |
| 0.5 ep | 1.0 ep | [93, 186] | 15% (asymmetric) |

**Rule of thumb:** keep the total warmup below 20% of `total_steps`.
Below 10% the head does not have enough time to stabilise before backbone
gradients flow in.  Above 30% you are wasting training budget on warmup.

The two values do not have to be equal.  A common choice is a shorter head
warmup (the head converges quickly) and a longer backbone ramp (the backbone
needs more time to adjust gradually).  For example `[372, 744]` warms up the
head for half an epoch and ramps the backbone over a full epoch.

In [ ]:
def load_cfg(filename):
    path = os.path.join(eomt_folder, 'configs', 'experiments', filename)
    with open(path) as f:
        cfg = yaml.safe_load(f)
    return cfg


def build_lora_model(cfg):
    """
    Build MaskClassificationLoRA from a YAML config.
    LoRA adapters are initialised to zero on top of the COCO checkpoint, so
    the model is a mathematical no-op at step 0.
    Freezing strategy is applied in each experiment cell after this call.
    """
    encoder_ft = ViT(img_size=640, backbone_name="vit_base_patch14_reg4_dinov2")
    network_ft  = EoMT(num_classes=19, encoder=encoder_ft,
                        num_q=200, num_blocks=3, masked_attn_enabled=True)
    return MaskClassificationLoRA(
        network=network_ft,
        img_size=(640, 640),
        num_classes=19,
        attn_mask_annealing_enabled=False,
        lr=cfg['lr'],
        llrd=cfg['llrd'],
        weight_decay=cfg['weight_decay'],
        poly_power=cfg['poly_power'],
        warmup_steps=cfg['warmup_steps'],
        ckpt_path=bin_path,
        load_ckpt_class_head=False,
        lora_r=cfg['lora_r'],
        lora_alpha=cfg['lora_alpha'],
        lora_target_modules=cfg['lora_target_modules'],
        lora_dropout=cfg['lora_dropout'],
    )


def build_trainer(cfg):
    wandb.finish()
    wandb.login(key=userdata.get('WANDDB-API-KEY'))
    run_name = cfg['experiment_name']
    return Trainer(
        max_epochs=cfg['max_epochs'],
        accelerator='auto',
        devices=1,
        precision='16-mixed',
        # fetch the accumulated gradient value and if not found adopt default 1
        accumulate_grad_batches=cfg.get('accumulate_grad_batches', 1),
        gradient_clip_val=cfg.get('gradient_clip_val', 0.01),
        gradient_clip_algorithm='norm', # preserves the direction of the gradient
        log_every_n_steps=10,
        num_sanity_val_steps=0,
        logger=WandbLogger(project='eomt-cityscapes-finetuning', name=run_name),
        callbacks=[
            LearningRateMonitor(logging_interval='step'),
            ModelCheckpoint(
                dirpath=os.path.join(project_root, 'checkpoints', run_name),
                filename='eomt-{epoch:02d}',
                save_top_k=2,
                monitor='metrics/val_iou_all',
                mode='max',
                save_last=True,
            ),
        ],
    )


def find_latest_ckpt(run_name):
    ckpt_dir = os.path.join(project_root, 'checkpoints', run_name)
    last = os.path.join(ckpt_dir, 'last.ckpt')
    if os.path.exists(last):
        print(f"Resuming from: {last}")
        return last
    ckpts = sorted(glob.glob(os.path.join(ckpt_dir, '*.ckpt')))
    if ckpts:
        print(f"Resuming from: {ckpts[-1]}")
        return ckpts[-1]
    print(f"No checkpoint for '{run_name}', starting from scratch.")
    return None


def print_trainable(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
cfg_shared = load_cfg('exp1_lora_head_only.yaml')
dm_train = CityscapesSemantic(
    path=data_path,
    batch_size=cfg_shared['batch_size'],
    num_workers=2,
    img_size=(640, 640),
)
dm_train.setup('fit')

## 4. Experiment 1: Head-Only (LoRA Frozen)

LoRA adapters are injected but then immediately frozen — they act purely as
structural scaffolding so the model has the same architecture as experiments 2
and 3, making the comparison fair.  Only the four prediction-head components
train: `class_head`, `mask_head`, `q`, `upscale`.

Because no encoder params have `requires_grad=True`, `configure_optimizers`
produces `num_backbone_params=0`.  The scheduler applies the non-backbone
(immediate) warmup to the head groups only.

In [ ]:
# If you change constanst in the yaml file remember to remove the last checkpoint
# !rm checkpoints/lora-head-only

In [ ]:
## We can expect a modest improvement compared to the pretrained COCO zero-shot model because
## the only thing training are the newly mounted class heads. The network still thinks in terms of COCO.
## Here, the network is basically learning a smooth, continuous version of the discrete mapping from the previous zero-shot step.
## Because the backbone is frozen, the capacity is limited.

## About the Loss oscillating around 10: This is completely NORMAL!
## EoMT uses a complex bipartite matching loss (classification + dice mask + cross-entropy mask).
## It is notoriously noisy and does not smoothly drop to near-zero like simple image classification.
## The absolute value of the training loss doesn't matter much here.

## The Plan:
## For just 10 epochs, early stopping is unnecessary and might interrupt a fair comparison.
## We will run all 3 experiments for exactly 10 epochs.
## The ONLY metric that matters is `metrics/val_iou_all` (Validation mIoU).
## We use this Experiment 1 (Head-only) as our lower-bound baseline to see if LoRA (Exp 2 & 3) actually helps!

In [ ]:
cfg1   = load_cfg('exp1_lora_head_only.yaml')
model1 = build_lora_model(cfg1)

# Freeze only the encoder (backbone originals + LoRA adapters).

for param in model1.network.encoder.parameters():
    param.requires_grad = False

print_trainable(model1)

trainer1 = build_trainer(cfg1)
trainer1.fit(model=model1, datamodule=dm_train,
             ckpt_path=find_latest_ckpt(cfg1['experiment_name']))


trainable params: 1,032,192 || all params: 87,929,856 || trainable%: 1.1739
Trainable: 6,677,780 / 94,607,636 (7.06%)


/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
INFO: Using 16bit Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightni

No checkpoint for 'lora-head-only', starting from scratch.


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: Loading `train_dataloader` to estimate number of stepping batches.
INFO:lightning.pytorch.utilities.rank_zero:Loading `train_dataloader` to estimate number of stepping batches.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


...

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/core/module.py:1333: Detected call of 
`lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite 
order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the 
first value of the learning rate schedule. See more details at 
https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate

INFO: mIoU: 48.4
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 48.4
INFO: mIoU: 59.1
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 59.1
INFO: mIoU: 63.3
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 63.3
INFO: 
Detected KeyboardInterrupt, attempting graceful shutdown ...
INFO:lightning.pytorch.utilities.rank_zero:
Detected KeyboardInterrupt, attempting graceful shutdown ...


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/call.py", line 49, in _call_and_handle_interrupt
    return trainer_fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/trainer.py", line 630, in _fit_impl
    self._run(model, ckpt_path=ckpt_path, weights_only=weights_only)
  File "/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/trainer.py", line 1079, in _run
    results = self._run_stage()
              ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/trainer.py", line 1123, in _run_stage
    self.fit_loop.run()
  File "/usr/local/lib/python3.12/dist-packages/lightning/pytorch/loops/fit_loop.py", line 217, in run
    self.advance()
  File "/usr/local/lib/python3.12/dist-packages/lightning/pytorch/loops/fit_loop.py", line 469, in advance
    self.epoch_loop.run(self._data_fetcher)
  File

TypeError: object of type 'NoneType' has no len()

In [ ]:
# Cliiping 0.01 grants loss decrease even with just classification heads worth maintaining

## 5. Experiment 2: LoRA on Decoder Blocks (9–11)

In EoMT the last `num_blocks` (= 3) ViT transformer blocks serve as the mask
decoder: the learnable query tokens are concatenated with patch tokens and
processed jointly in these blocks.  Enabling LoRA only in blocks 9–11 lets the
model adapt its highest-level representations without touching the earlier blocks
that encode more general visual features.

`configure_optimizers` sees backbone params from blocks 9–11 only.  Their LLRD
exponents are 0 (block 11), 1 (block 10), and 2 (block 9), so all three receive
a relatively high LR — they are the most task-relevant layers.

In [ ]:
cfg2   = load_cfg('exp2_lora_decoder_blocks.yaml')
model2 = build_lora_model(cfg2)

# Freeze the entire encoder (backbone originals + all LoRA adapters).
# Heads remain trainable — they are never frozen.
for param in model2.network.encoder.parameters():
    param.requires_grad = False

# Re-enable LoRA adapters in the last 3 blocks (the decoder blocks only).
num_blocks    = len(model2.network.encoder.backbone.blocks)  # 12
decoder_start = num_blocks - model2.network.num_blocks       # 9
for name, param in model2.network.encoder.named_parameters():
    parts = name.split('.')
    for i, part in enumerate(parts):
        if part == 'blocks' and i + 1 < len(parts):
            try:
                if int(parts[i + 1]) >= decoder_start:
                    param.requires_grad = True
            except ValueError:
                pass

print_trainable(model2)

trainer2 = build_trainer(cfg2)
trainer2.fit(model=model2, datamodule=dm_train,
             ckpt_path=find_latest_ckpt(cfg2['experiment_name']))


trainable params: 1,032,192 || all params: 87,929,856 || trainable%: 1.1739
Trainable: 28,204,052 / 94,607,636 (29.81%)


/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.


epoch,▁▁▁▁▁▁▁▁▁▁▁▃▃▃▃▃▃▃▃▃▃▃▃▃▃▆▆▆▆▆▆▆▆███████
losses/train_loss_cross_entropy,███▇▇▆▅▃▃▃▂▂▂▂▂▂▂▁▁▂▁▁▁▁▁▂▁▁▁▁▂▁▂▁▁▁▁▁▁▁
losses/train_loss_cross_entropy_block_-1,███▇▇▅▅▄▃▃▃▂▂▂▂▁▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
losses/train_loss_cross_entropy_block_-2,███▇▇▆▆▅▄▃▃▂▂▂▂▂▁▁▁▂▁▂▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
losses/train_loss_cross_entropy_block_-3,████▇▆▆▅▅▄▃▃▃▃▂▂▁▂▁▂▁▁▂▁▂▂▂▁▂▂▂▁▁▁▁▁▁▁▁▁
losses/train_loss_dice,█▆▆▅█▄▆▄▃▅▄▆▃▄▄▃▅▁▅▄▆▄▃▄▄▂▄▅▄▅▂▁▃▃▅▄▃▃▂▃
losses/train_loss_dice_block_-1,█▅▆█▄▄▆▅▄▄▄▆▆▇▄▃▄▁▄▂▅▄▂▅▂▅▄▅▂▂▃▂▅▃▃▂▄▃▃▄
losses/train_loss_dice_block_-2,█▆▆█▄▆▅▅▄▄▅▄▆▃▄▃▅▁▅▃▅▄▄▅▂▆▅▅▄▃▁▃▃▅▄▂▃▃▃▄
losses/train_loss_dice_block_-3,▅▆▅▆█▄▆▆▇▄▃▅▄▅▅▂▄▇▄▄▄▄▁▅▁▄▂▃▄▅▄▃▃▂▁▂▂▃▁▄
losses/train_loss_mask,█▆▅▅▂▃▄▃▃▃▂▃▃▄▂▃▄▁▁▁▄▂▂▄▃▃▁▃▄▃▂▄▁▁▄▅▃▄▂▃
+28,...


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
INFO: Using 16bit Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightni

No checkpoint for 'lora-decoder-blocks', starting from scratch.


...

┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type                   ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ network   │ EoMT                   │ 94.6 M │ train │     0 │
│ 1 │ criterion │ MaskClassificationLoss │      0 │ train │     0 │
│ 2 │ metrics   │ ModuleList             │      0 │ train │     0 │
└───┴───────────┴────────────────────────┴────────┴───────┴───────┘

Trainable params: 28.2 M                                                                                           
Non-trainable params: 66.4 M                                                                                       
Total params: 94.6 M                                                                                               
Total estimated model params size (MB): 378.431                                                                    
Modules in train mode: 666                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/core/module.py:1333: Detected call of 
`lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite 
order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the 
first value of the learning rate schedule. See more details at 
https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate

INFO: mIoU: 53.5
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 53.5
INFO: mIoU: 68.6
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 68.6
INFO: mIoU: 72.5
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 72.5
INFO: mIoU: 73.5
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 73.5
INFO: mIoU: 75.0
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 75.0
INFO: mIoU: 74.8
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 74.8
INFO: mIoU: 75.2
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 75.2
INFO: mIoU: 74.8
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 74.8
INFO: mIoU: 75.6
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 75.6
INFO: mIoU: 75.8
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 75.8
INFO: `Trainer.fit` stopped: `max_epochs=10` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=10` reached.


## 6. Experiment 3: Full LoRA (All 12 Blocks)

LoRA adapters in all 12 blocks are trainable.  This is the default behaviour of
`MaskClassificationLoRA` — no manual freezing is needed after construction.

`configure_optimizers` applies LLRD across all 12 blocks:
```
block  0  LR = 1e-4 × 0.9^11 ≈ 3.1e-5   (low — general texture/edge features)
block 11  LR = 1e-4 × 0.9^0  = 1.0e-4   (full — task-specific decoder features)
```
The two-stage warmup freezes all backbone groups for the first 125 optimizer steps
(~0.67 epochs) while the head stabilises, then ramps the backbone LR over the next
125 steps.

In [ ]:
cfg3   = load_cfg('exp3_lora_all_blocks.yaml')
model3 = build_lora_model(cfg3)
# No manual requires_grad changes — MaskClassificationLoRA already sets
# LoRA adapters trainable and backbone original weights frozen by default.
print_trainable(model3)

trainer3 = build_trainer(cfg3)
trainer3.fit(model=model3, datamodule=dm_train,
             ckpt_path=find_latest_ckpt(cfg3['experiment_name']))

trainable params: 1,032,192 || all params: 87,929,856 || trainable%: 1.1739
Trainable: 7,709,972 / 94,607,636 (8.15%)


/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.


epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▅▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇▇█████
losses/train_loss_cross_entropy,█▅▂▂▂▂▁▁▂▁▂▁▁▁▂▁▁▂▂▁▂▁▁▁▂▁▁▁▂▂▁▁▁▁▁▁▁▁▂▁
losses/train_loss_cross_entropy_block_-1,█▃▃▁▂▂▁▂▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▂▁▁
losses/train_loss_cross_entropy_block_-2,█▅▅▃▂▁▂▂▂▂▁▂▁▁▁▂▁▂▁▁▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁
losses/train_loss_cross_entropy_block_-3,█▅▄▄▂▃▂▂▃▁▂▂▂▁▁▁▂▂▁▂▂▂▁▁▂▂▂▂▂▂▂▂▁▂▁▁▁▁▁▂
losses/train_loss_dice,██▆▄▃▅▃▄▃█▃▆▃▁▇▅▄█▅▄▃▁▄▂▃▆▄▄▃▆▂▄▆▄▅▇▂▂▂▁
losses/train_loss_dice_block_-1,█▇▇▅▅▆▅▆▆▆▄▅▃▇▄▄▅▇▄▆▂█▄▇▅▄▄▆▃▅▄▂▆▅▁▄▆▅▅▂
losses/train_loss_dice_block_-2,█▅▆▃▆▆▇▄▅▄▆▆▃▆▇▆█▃▅▇▂▆▃▇▃▆▄▅▆▂▄▄▂▅▃▁▆▂▄▅
losses/train_loss_dice_block_-3,█▆▇▅▆▅▂▅▄▆▅▆█▆▂▄█▃▃▅▅▄▅▆▆▃▆▄▃▃▃▅▄▄▅▁▂▆▅▇
losses/train_loss_mask,█▃▃▃▄▃▅▃▃▃▄▁▃▃▃▂▂▄▂▃▄▄▂▄▄▁▅▄▃▃▁▄▂▂▁▄▄▃▅▃
+88,...


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
INFO: Using 16bit Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightni

No checkpoint for 'lora-all-blocks', starting from scratch.


...

┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type                   ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ network   │ EoMT                   │ 94.6 M │ train │     0 │
│ 1 │ criterion │ MaskClassificationLoss │      0 │ train │     0 │
│ 2 │ metrics   │ ModuleList             │      0 │ train │     0 │
└───┴───────────┴────────────────────────┴────────┴───────┴───────┘

Trainable params: 7.7 M                                                                                            
Non-trainable params: 86.9 M                                                                                       
Total params: 94.6 M                                                                                               
Total estimated model params size (MB): 378.431                                                                    
Modules in train mode: 666                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/core/module.py:1333: Detected call of 
`lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite 
order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the 
first value of the learning rate schedule. See more details at 
https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate

INFO: mIoU: 49.5
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 49.5
INFO: mIoU: 64.3
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 64.3
INFO: mIoU: 67.1
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 67.1
INFO: mIoU: 67.9
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 67.9
INFO: mIoU: 70.2
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 70.2
INFO: mIoU: 70.9
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 70.9
INFO: mIoU: 71.1
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 71.1
INFO: mIoU: 71.1
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 71.1
INFO: mIoU: 72.0
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 72.0
INFO: mIoU: 71.6
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 71.6
INFO: `Trainer.fit` stopped: `max_epochs=10` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=10` reached.


## 7. Results

In [ ]:
results = {
    'COCO zero-shot':              zeroshot_miou,
    'Head-only  (exp1)':           None,   # wandb: lora-head-only
    'Decoder LoRA  (exp2)':        None,   # wandb: lora-decoder-blocks
    'Full LoRA  (exp3)':           None,   # wandb: lora-all-blocks
    'Cityscapes pretrained':       None,
}

print(f"{'Model':<35} {'val mIoU (%)':>12}")
print('-' * 49)
for name, miou in results.items():
    val = f'{miou:.2f}' if miou is not None else 'pending'
    print(f'{name:<35} {val:>12}')

Model                               val mIoU (%)
-------------------------------------------------
COCO zero-shot                             45.91
Head-only  (exp1)                        pending
Decoder LoRA  (exp2)                     pending
Full LoRA  (exp3)                        pending
Cityscapes pretrained                    pending
